In [1]:
import pandas as pd
import numpy as np
import re
import zipfile

zip_path = "./original_kaggle_data_download_archive.zip"

with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())  # See what files are inside
    f1 = pd.read_csv(z.open("1429_1.csv"))
    f2 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"))
    f3 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"))

['1429_1.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv']


/tmp/ipykernel_708/2601716990.py:10: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  f1 = pd.read_csv(z.open("1429_1.csv"))


In [2]:
"""
Data preparation for the Automated Customer Reviews project.
Merges the 3 raw Datafiniti/Amazon review exports, deduplicates, cleans the
corrupted `name` field, maps star ratings to sentiment classes, and writes
a single clean dataset for downstream modeling.
"""

OUT_PATH = "clean_reviews.csv"

print(f"f1 {f1.shape}, f2 {f2.shape}, f3 {f3.shape}")

# ---- Keep only columns relevant to NLP tasks ----
keep_cols = ["id", "name", "asins", "brand", "categories", "primaryCategories",
             "reviews.date", "reviews.rating", "reviews.text", "reviews.title"]

for df in (f1, f2, f3):
    for c in keep_cols:
        if c not in df.columns:
            df[c] = np.nan

merged = pd.concat([f1[keep_cols], f2[keep_cols], f3[keep_cols]], ignore_index=True)
print(f"merged raw {merged.shape}")

# ---- Deduplicate: same product + review text + rating ----
merged = merged.dropna(subset=["reviews.text", "reviews.rating"])
merged = merged.drop_duplicates(subset=["id", "reviews.text", "reviews.rating"])
print(f"after dedup {merged.shape}")
print(f"unique product IDs: {merged['id'].nunique()}")
print(f"unique product names (raw): {merged['name'].nunique()}")

# ---- Fix corrupted `name` field (two names concatenated together) ----
def clean_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    # Datafiniti corruption pattern: "Name A,,,,Name B" or names glued with commas
    # Keep first product-name segment (before a run of 2+ commas, or before an
    # obvious second product name starting after a comma cluster)
    parts = re.split(r",{2,}", name)
    first = parts[0].strip()
    return first

merged["name_clean"] = merged["name"].apply(clean_name)
print(f"unique product names (cleaned): {merged['name_clean'].nunique()}")



f1 (34660, 21), f2 (5000, 24), f3 (28332, 24)
merged raw (67992, 10)
after dedup (59743, 10)
unique product IDs: 89
unique product names (raw): 119
unique product names (cleaned): 109


In [3]:
# ---- Sentiment mapping ----
def map_sentiment(rating):
    if pd.isna(rating):
        return np.nan
    rating = int(rating)
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

merged["sentiment"] = merged["reviews.rating"].apply(map_sentiment)
merged = merged.dropna(subset=["sentiment"])

# ---- Combine title + text as model input ----
merged["review_text_full"] = (
    merged["reviews.title"].fillna("") + ". " + merged["reviews.text"].fillna("")
).str.strip()
merged = merged[merged["review_text_full"].str.len() > 3]

print("\nFinal dataset:")
print(f"  rows: {len(merged)}")
print(f"  unique product IDs: {merged['id'].nunique()}")
print(f"  unique cleaned product names: {merged['name_clean'].nunique()}")
print("\nSentiment class distribution:")
print(merged["sentiment"].value_counts())
print(merged["sentiment"].value_counts(normalize=True).round(3))

merged.to_csv(OUT_PATH, index=False)
print(f"\nSaved -> {OUT_PATH}")


Final dataset:
  rows: 59743
  unique product IDs: 89
  unique cleaned product names: 109

Sentiment class distribution:
sentiment
positive    54841
neutral      2588
negative     2314
Name: count, dtype: int64
sentiment
positive    0.918
neutral     0.043
negative    0.039
Name: proportion, dtype: float64

Saved -> clean_reviews.csv



<hr>

## Create a unique product table

In [4]:
# ---- Clustering only the unique product names ----

unique_products = merged["name_clean"].drop_duplicates()

print(len(unique_products))

110


## Convert to TF-IDF vectors

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2)
)
unique_products = pd.Series(unique_products).dropna().astype(str)
X = vectorizer.fit_transform(unique_products)

## Run KMeans

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np
import joblib
import pandas as pd

# ---------------------------------------------------------
# 1. Split for evaluation purposes
# ---------------------------------------------------------
X_train, X_test = train_test_split(X.toarray(), test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 2. Fit KMeans on train, evaluate on train + test
# ---------------------------------------------------------
kmeans = KMeans(n_clusters=7, random_state=42, n_init=20)
kmeans.fit(X_train)

def test_inertia(model, X):
    _, dists = pairwise_distances_argmin_min(X, model.cluster_centers_)
    return np.sum(dists ** 2)

def evaluate_kmeans(model, X, labels, set_name, is_train):
    metrics = {
        "inertia": model.inertia_ if is_train else test_inertia(model, X),
        "silhouette_score": silhouette_score(X, labels),
        "calinski_harabasz_score": calinski_harabasz_score(X, labels),
        "davies_bouldin_score": davies_bouldin_score(X, labels),
    }
    print(f"--- {set_name} metrics ---")
    for name, val in metrics.items():
        print(f"{name}: {val:.4f}")
    print()
    return metrics

train_labels = kmeans.labels_
test_labels = kmeans.predict(X_test)

train_metrics = evaluate_kmeans(kmeans, X_train, train_labels, "Train", is_train=True)
test_metrics = evaluate_kmeans(kmeans, X_test, test_labels, "Test", is_train=False)

# ---------------------------------------------------------
# 3. Fit the FINAL model on all data (your original approach)
#    — standard practice once you've validated cluster quality on the split
# ---------------------------------------------------------
final_kmeans = KMeans(n_clusters=7, random_state=42, n_init=20)
labels = final_kmeans.fit_predict(X)

products = pd.DataFrame({
    "product": unique_products,
    "cluster": labels
})

# ---------------------------------------------------------
# 4. Save the final model + vectorizer
# ---------------------------------------------------------
joblib.dump(final_kmeans, "kmeans_product_category_model.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
print("Saved: kmeans_product_category_model.joblib, tfidf_vectorizer.joblib")

# --- To reload later ---
# kmeans_loaded = joblib.load("kmeans_product_category_model.joblib")
# vectorizer_loaded = joblib.load("tfidf_vectorizer.joblib")
# new_cluster = kmeans_loaded.predict(vectorizer_loaded.transform(["some new product name"]))

--- Train metrics ---
inertia: 61.8995
silhouette_score: 0.0938
calinski_harabasz_score: 4.1190
davies_bouldin_score: 2.9653

--- Test metrics ---
inertia: 17.9906
silhouette_score: 0.0662
calinski_harabasz_score: 1.5847
davies_bouldin_score: 1.8576

Saved: kmeans_product_category_model.joblib, tfidf_vectorizer.joblib


<hr>